# Lab 02 - Generadores pseudoaleatorios y transformada inversa

**Modelación y Simulación**  
En este notebook se implementan los cinco ejercicios del laboratorio. Se usa $\alpha=0.05$ en todas las pruebas de hipótesis.

In [ ]:
import gc
import time
from math import ceil, log

import matplotlib.pyplot as plt
import nistrng
import numpy as np
from scipy import stats

ALPHA = 0.05
np.set_printoptions(precision=4, suppress=True)

## Funciones auxiliares

La regla de decisión usada es: no se rechaza $H_0$ cuando el p-value es mayor o igual a $\alpha$.

In [ ]:
def conclusion(p_value, alpha=ALPHA):
    return "No se rechaza H0" if p_value >= alpha else "Se rechaza H0"


def describe_uniform(sample, label, bins=20):
    ks = stats.kstest(sample, "uniform")
    counts, _ = np.histogram(sample, bins=bins, range=(0, 1))
    chi = stats.chisquare(counts)
    print(f"{label}: n={len(sample)}, media={sample.mean():.5f}, varianza={sample.var(ddof=1):.5f}")
    print(f"  KS: estadístico={ks.statistic:.5f}, p-value={ks.pvalue:.5g}. {conclusion(ks.pvalue)}")
    print(f"  Chi-cuadrado: estadístico={chi.statistic:.5f}, p-value={chi.pvalue:.5g}. {conclusion(chi.pvalue)}")
    plt.figure(figsize=(7, 3.5))
    plt.hist(sample, bins=bins, range=(0, 1), density=True, edgecolor="black", alpha=0.75)
    plt.axhline(1, color="red", linestyle="--", label="densidad teórica")
    plt.title(label)
    plt.xlabel("valor")
    plt.ylabel("densidad")
    plt.legend()
    plt.show()

# 1. Generador congruencial lineal (LCG)

Se genera primero la muestra discreta $x_1,\ldots,x_N$ y luego se normaliza como $u_i=x_i/m$ para obtener valores en $[0,1)$.

In [ ]:
def lcg(seed, a, c, m, n):
    values = np.empty(n, dtype=np.int64)
    x = seed
    for i in range(n):
        x = (a * x + c) % m
        values[i] = x
    return values


# Comprobaciones pequeñas antes de los experimentos.
check = lcg(1, 5, 1, 16, 10)
assert len(check) == 10
assert np.all((check >= 0) & (check < 16))

lcg_cases = [
    {"name": "LCG 1: Numerical Recipes", "seed": 12345, "a": 1664525, "c": 1013904223, "m": 2**32},
    {"name": "LCG 2: Park-Miller", "seed": 12345, "a": 16807, "c": 0, "m": 2**31 - 1},
]
N_UNIFORM = 10_000
lcg_uniform_samples = {}

for params in lcg_cases:
    discrete = lcg(params["seed"], params["a"], params["c"], params["m"], N_UNIFORM)
    uniform = discrete / params["m"]
    lcg_uniform_samples[params["name"]] = uniform
    print(f"\n{params['name']}: a={params['a']}, c={params['c']}, m={params['m']}, N={N_UNIFORM}")
    print("Primeros 10 valores discretos:", discrete[:10])
    describe_uniform(uniform, params["name"])

# 2. Mersenne Twister

Se implementa MT19937 con el tamaño de estado estándar de 624 enteros de 32 bits. Para el experimento se convierte cada entero a un valor uniforme dividiendo entre $2^{32}$.

In [ ]:
class MersenneTwister:
    def __init__(self, seed=5489):
        self.n, self.m = 624, 397
        self.matrix_a = 0x9908B0DF
        self.upper_mask = 0x80000000
        self.lower_mask = 0x7FFFFFFF
        self.state = np.zeros(self.n, dtype=np.uint32)
        self.state[0] = seed & 0xFFFFFFFF
        for i in range(1, self.n):
            self.state[i] = (1812433253 * (int(self.state[i - 1]) ^ (int(self.state[i - 1]) >> 30)) + i) & 0xFFFFFFFF
        self.index = self.n

    def twist(self):
        for i in range(self.n):
            x = (int(self.state[i]) & self.upper_mask) + (int(self.state[(i + 1) % self.n]) & self.lower_mask)
            x_a = x >> 1
            if x & 1:
                x_a ^= self.matrix_a
            self.state[i] = int(self.state[(i + self.m) % self.n]) ^ x_a
        self.index = 0

    def random_uint32(self):
        if self.index >= self.n:
            self.twist()
        y = int(self.state[self.index])
        y ^= y >> 11
        y ^= (y << 7) & 0x9D2C5680
        y ^= (y << 15) & 0xEFC60000
        y ^= y >> 18
        self.index += 1
        return y & 0xFFFFFFFF

    def random(self):
        return self.random_uint32() / 2**32


mt_check = MersenneTwister(7)
assert 0 <= mt_check.random() < 1

mt = MersenneTwister(2026)
mt_uniform = np.fromiter((mt.random() for _ in range(N_UNIFORM)), dtype=float, count=N_UNIFORM)
describe_uniform(mt_uniform, "Mersenne Twister MT19937")

# 3. Tests NIST SP 800-22

Se generan 1,000,000 bits por generador. La biblioteca `nistrng` implementa los 15 tests de la batería SP 800-22 Rev. 1a. Un p-value menor que $0.05$ marca un rechazo para esa prueba.

In [ ]:
def bits_from_lcg(params, n_bits):
    # Usar el bit más significativo evita depender solamente de la paridad del LCG.
    values = lcg(params["seed"], params["a"], params["c"], params["m"], n_bits)
    return ((values / params["m"]) >= 0.5).astype(np.int64)


def bits_from_mt(seed, n_bits):
    generator = MersenneTwister(seed)
    return np.fromiter((generator.random() >= 0.5 for _ in range(n_bits)), dtype=np.int64, count=n_bits)


def nist_table(bits, generator_name):
    start = time.perf_counter()
    rows = []
    # Se ejecuta una prueba a la vez para no acumular memoria en secuencias largas.
    for name in nistrng.SP800_22R1A_BATTERY:
        item = nistrng.run_by_name_battery(name, bits, nistrng.SP800_22R1A_BATTERY, check_eligibility=True)
        if item is None:
            rows.append((generator_name, "No aplicable", np.nan, "No aplicable"))
        else:
            result, elapsed = item
            rows.append((generator_name, result.name, result.score, "Pasa" if result.passed else "No pasa"))
        gc.collect()
    print(f"{generator_name}: {time.perf_counter() - start:.1f} segundos")
    return rows


N_BITS = 1_000_000
lcg_bits = bits_from_lcg(lcg_cases[0], N_BITS)
mt_bits = bits_from_mt(2026, N_BITS)
assert len(lcg_bits) == len(mt_bits) == N_BITS
assert set(np.unique(lcg_bits)).issubset({0, 1})
assert set(np.unique(mt_bits)).issubset({0, 1})

nist_rows = nist_table(lcg_bits, "LCG 1") + nist_table(mt_bits, "Mersenne Twister")
print("\nGenerador | Test | p-value | Resultado")
print("-" * 75)
for generator, test, p_value, result in nist_rows:
    value = "-" if np.isnan(p_value) else f"{p_value:.5g}"
    print(f"{generator:16} | {test:36} | {value:>9} | {result}")

for generator in ("LCG 1", "Mersenne Twister"):
    passed = sum(row[3] == "Pasa" for row in nist_rows if row[0] == generator)
    evaluated = sum(row[3] != "No aplicable" for row in nist_rows if row[0] == generator)
    print(f"{generator}: {passed}/{evaluated} pruebas aprobadas.")

# 4. Comparación de muestras geométricas

`scipy.stats.geom` genera la muestra teórica. Para la muestra empírica se usa $X=\lceil\log(1-U)/\log(1-p)\rceil$. Se comparan frecuencias por chi-cuadrado y las dos muestras con KS.

In [ ]:
def inverse_geometric(u, p):
    u = np.clip(np.asarray(u), np.finfo(float).eps, 1 - np.finfo(float).eps)
    return np.ceil(np.log1p(-u) / np.log1p(-p)).astype(int)


def compare_geometric(p=0.35, n=10_000, seed=2026):
    rng = np.random.default_rng(seed)
    theoretical = stats.geom.rvs(p, size=n, random_state=rng)
    empirical = inverse_geometric(rng.random(n), p)
    assert len(theoretical) == len(empirical) == n
    assert empirical.min() >= 1

    upper = max(np.quantile(theoretical, 0.99), np.quantile(empirical, 0.99)).astype(int)
    bins = np.arange(1, upper + 2)
    observed = np.histogram(empirical, bins=np.append(bins, np.inf))[0]
    expected = np.histogram(theoretical, bins=np.append(bins, np.inf))[0]
    chi_statistic, chi_pvalue, _, _ = stats.chi2_contingency(
        np.vstack([observed, expected]), correction=False
    )
    ks = stats.ks_2samp(theoretical, empirical)
    print(f"Geométrica(p={p}), N={n}")
    print(f"Chi-cuadrado: p-value={chi_pvalue:.5g}. {conclusion(chi_pvalue)}")
    print(f"KS: p-value={ks.pvalue:.5g}. {conclusion(ks.pvalue)}")
    plt.figure(figsize=(7, 3.5))
    plt.hist(theoretical, bins=range(1, upper + 2), alpha=0.55, density=True, label="teórica")
    plt.hist(empirical, bins=range(1, upper + 2), alpha=0.55, density=True, label="empírica")
    plt.title("Muestras geométricas")
    plt.xlabel("x")
    plt.legend()
    plt.show()


assert np.array_equal(inverse_geometric(np.array([0.1, 0.9]), 0.5), np.array([1, 4]))
compare_geometric()

# 5. Comparación de muestras normales

La transformada inversa usa `norm.ppf`, que corresponde a $X=\mu+\sigma\Phi^{-1}(U)$. Se usan los mismos contrastes de chi-cuadrado y KS.

In [ ]:
def inverse_normal(u, mu, sigma):
    u = np.clip(np.asarray(u), np.finfo(float).eps, 1 - np.finfo(float).eps)
    return stats.norm.ppf(u, loc=mu, scale=sigma)


def compare_normal(mu=10, sigma=2, n=10_000, seed=2026):
    rng = np.random.default_rng(seed)
    theoretical = stats.norm.rvs(loc=mu, scale=sigma, size=n, random_state=rng)
    empirical = inverse_normal(rng.random(n), mu, sigma)
    assert len(theoretical) == len(empirical) == n

    bins = np.linspace(min(theoretical.min(), empirical.min()), max(theoretical.max(), empirical.max()), 21)
    observed = np.histogram(empirical, bins=bins)[0]
    expected = np.histogram(theoretical, bins=bins)[0]
    chi_statistic, chi_pvalue, _, _ = stats.chi2_contingency(
        np.vstack([observed, expected]), correction=False
    )
    ks = stats.ks_2samp(theoretical, empirical)
    print(f"Normal(mu={mu}, sigma={sigma}), N={n}")
    print(f"Chi-cuadrado: p-value={chi_pvalue:.5g}. {conclusion(chi_pvalue)}")
    print(f"KS: p-value={ks.pvalue:.5g}. {conclusion(ks.pvalue)}")
    plt.figure(figsize=(7, 3.5))
    plt.hist(theoretical, bins=30, alpha=0.55, density=True, label="teórica")
    plt.hist(empirical, bins=30, alpha=0.55, density=True, label="empírica")
    plt.title("Muestras normales")
    plt.xlabel("x")
    plt.legend()
    plt.show()


assert np.isclose(inverse_normal(np.array([0.5]), 3, 2)[0], 3)
compare_normal()

## Conclusión

Las pruebas de uniformidad y la batería NIST permiten contrastar ambos generadores. La comparación final se basa en los p-values impresos: cuando ambos p-values son mayores o iguales a 0.05, no hay evidencia estadística para afirmar que la muestra empírica y la teórica provienen de distribuciones distintas.